In [ ]:
import torch
import numpy as np
import cv2
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.modeling import build_model
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.structures import ImageList

# --- Configuration Setup ---
def setup_model():
    # Load Config for Mask R-CNN R101 (as per paper/notebook)
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_101_FPN_3x.yaml"))
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5  # Threshold
    cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/mask_rcnn_R_101_FPN_3x.yaml")

    # Force CPU for demonstration if CUDA not available (change to 'cuda' if you have GPU)
    cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    # Build model explicitly to access internal layers
    model = build_model(cfg)
    checkpointer = DetectionCheckpointer(model)
    checkpointer.load(cfg.MODEL.WEIGHTS)
    model.eval()
    return model, cfg

model, cfg = setup_model()

# --- Extraction Function ---
def get_manm_embeddings(image_path, model, cfg, max_regions=100):
    """
    Extracts 1024-d visual features and 6-d geometric features
    padded to a fixed sequence length of 100.
    """
    # 1. Load and Preprocess Image
    img = cv2.imread(image_path)
    if img is None:
        return None, None

    height, width = img.shape[:2]
    img_tensor = torch.as_tensor(img.astype("float32").transpose(2, 0, 1))
    inputs = [{"image": img_tensor, "height": height, "width": width}]

    with torch.no_grad():
        images = model.preprocess_image(inputs)  # Normalization & resizing

        # 2. Extract Feature Maps (Backbone)
        features = model.backbone(images.tensor)

        # 3. Generate Proposals (RPN)
        proposals, _ = model.proposal_generator(images, features, None)
        proposal = proposals[0] # Take first image in batch

        # 4. Filter Proposals (Top-K)
        # We need exactly max_regions (100).
        # Detectron2 usually returns 1000. We sort by score.
        scores = proposal.objectness_logits
        topk_idx = torch.topk(scores, min(len(scores), max_regions)).indices
        selected_boxes = proposal.proposal_boxes[topk_idx]

        # 5. Extract ROI Features (Box Pooler) -> 1024 dim
        # Get feature maps keys (p2, p3, p4, p5)
        feature_list = [features[f] for f in cfg.MODEL.ROI_HEADS.IN_FEATURES]

        # Pool features for the selected boxes
        box_features = model.roi_heads.box_pooler(feature_list, [selected_boxes])

        # Flatten and pass through FC layers (FC1 -> ReLU -> FC2) to get 1024-d embedding
        # Note: box_head structure depends on config. Standard ResNet head has these layers.
        box_features = model.roi_heads.box_head.flatten(box_features)
        box_features = model.roi_heads.box_head.fc1(box_features)
        box_features = model.roi_heads.box_head.fc_relu1(box_features)
        box_features = model.roi_heads.box_head.fc2(box_features)
        # Output shape: (N, 1024)

        # 6. Calculate Position Embeddings (6-dim) -> [x1, y1, x2, y2, area, ratio]
        # Paper requires "normalized" values [cite: 393]
        boxes_tensor = selected_boxes.tensor # (N, 4) -> x1, y1, x2, y2

        # Normalize coordinates
        x1 = boxes_tensor[:, 0] / width
        y1 = boxes_tensor[:, 1] / height
        x2 = boxes_tensor[:, 2] / width
        y2 = boxes_tensor[:, 3] / height

        # Calculate Area and Aspect Ratio
        box_w = (boxes_tensor[:, 2] - boxes_tensor[:, 0])
        box_h = (boxes_tensor[:, 3] - boxes_tensor[:, 1])
        img_area = width * height

        norm_area = (box_w * box_h) / img_area
        aspect_ratio = box_w / (box_h + 1e-6) # Avoid div by zero

        # Stack to create 6-dim vector [cite: 393]
        pos_embeddings = torch.stack([x1, y1, x2, y2, norm_area, aspect_ratio], dim=1)

        # 7. Pad or Truncate to Fixed Size (100 regions) [cite: 246, 258]
        # Create zero tensors
        final_visual = torch.zeros((max_regions, 1024))
        final_pos = torch.zeros((max_regions, 6))

        num_boxes = box_features.shape[0]
        limit = min(num_boxes, max_regions)

        final_visual[:limit] = box_features[:limit].cpu()
        final_pos[:limit] = pos_embeddings[:limit].cpu()

        return final_visual.numpy(), final_pos.numpy()

# --- Usage Example ---
img_path = "path_to_your_image.jpg" # Replace with valid path
visual_emb, pos_emb = get_manm_embeddings(img_path, model, cfg)

if visual_emb is not None:
    print(f"Visual Embeddings Shape: {visual_emb.shape}") # Should be (100, 1024)
    print(f"Position Embeddings Shape: {pos_emb.shape}") # Should be (100, 6)

    # Save for training
    # np.save("visual_feats.npy", visual_emb)
    # np.save("pos_feats.npy", pos_emb)